# 01 · NumPy vs CuPy 벤치마크 기본

> **CuPy 2일 집중 코스 — Day 1 / 단원 1 (GPU 컴퓨팅과 CuPy 개론)**

GPU가 *언제* 빠른지를 **직접 측정**으로 배웁니다. 올바른 벤치마킹 방법을 익히고,
문제 크기에 따른 손익분기점과 전송 비용, 벡터화의 효과를 확인합니다.

## 학습 목표
- GPU 비동기 특성을 고려한 **올바른 벤치마킹**(워밍업·동기화·`benchmark`)을 수행한다.
- 문제 크기에 따라 GPU 이득이 달라지는 **손익분기점(break-even)** 을 찾는다.
- 전송 포함 **end-to-end** 비용과 연산만의 비용을 구분한다.
- **벡터화**가 (CPU·GPU 모두) 왜 필수인지 수치로 확인한다.

## 목차
1. [GPU 벤치마킹의 원칙](#1)
   - [1.1 일회성 오버헤드 & %gpu_timeit](#1)
2. [크기별 성능 비교 — 손익분기점](#2)
3. [전송 포함 end-to-end 비용](#3)
4. [벡터화의 힘](#4)
5. [연습문제](#5)
6. [체크포인트](#6)

> 참고: `course_utils.bench/compare/print_bench`는 모두 `cupyx.profiler.benchmark` 기반입니다. (단원 0에서 소개)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare
print_env()

<a id="1"></a>
## 1. GPU 벤치마킹의 원칙

GPU 연산은 **비동기**입니다. 그래서 `time.perf_counter()`로 한 번 재면 부정확합니다. 올바른 측정의 3원칙:
1. **워밍업(warmup)**: 첫 호출엔 커널 컴파일·캐시 등 1회성 비용이 섞입니다. 몇 번 버린 뒤 측정.
2. **동기화(synchronize)**: 커널 완료를 기다린 뒤 시간을 읽습니다.
3. **반복 평균**: 여러 번 반복해 평균±표준편차로 봅니다.

`cupyx.profiler.benchmark`(= `course_utils.bench`)가 이 셋을 모두 처리합니다.
공정한 CPU↔GPU 비교를 위해 **wall-clock(`cpu_times`)** 을 기준으로 삼습니다(전송·동기화 포함, end-to-end 관점).

In [ ]:
# benchmark는 CPU(wall-clock)와 GPU(kernel) 시간을 함께 보고합니다.
a = cp.random.random(10_000_000, dtype=cp.float32)
r = bench(lambda: (a * 1.1 + 2.0).sum(), n_repeat=20, n_warmup=3, name='fused_sum')
print_bench(r)
print(r)   # benchmark 객체의 기본 출력(반복 통계 포함)

### 1.1 일회성 오버헤드 & `%gpu_timeit`

워밍업이 필요한 이유는 **일회성 오버헤드** 때문입니다(공식 문서 권고).
- 프로세스 첫 CUDA 호출 시 **컨텍스트 초기화**(수 초).
- 인자 shape/dtype별 **커널 JIT 컴파일**(이후 `~/.cupy/kernel_cache`에 캐시되어 재사용).

주피터/IPython에서는 `cupyx.profiler` 확장의 **`%gpu_timeit`** 매직으로 간단히 잴 수 있습니다(`benchmark`와 동일 옵션 `-n`,`-w`).

```python
%load_ext cupyx.profiler
%gpu_timeit -n 20 (cp.random.random(1_000_000, dtype=cp.float32) ** 2).sum()

%%gpu_timeit
x = cp.random.random((2000, 2000), dtype=cp.float32)
y = x @ x.T
```

<a id="2"></a>
## 2. 크기별 성능 비교 — 손익분기점

GPU는 **커널 런치 오버헤드**(수~수십 µs)와 **전송 비용**이 있어, 데이터가 작으면 CPU보다 느릴 수 있습니다.
GPU가 이기려면 **충분한 작업량(workload)** 이 필요합니다. 아래 도해의 직관:
연산이 `O(N)`이면 이득을 보려면 데이터가 커야 하고, 연산량이 `O(N)`보다 크면 더 적은 데이터로도 이득을 봅니다.

<img src="images/figures/new_latency_bandwidth.png" width="640">



In [ ]:
# 같은 연산 (a*1.1+b).sum() 을 크기별로 CPU vs GPU 비교 (wall-clock)
sizes = [10_000, 100_000, 1_000_000, 10_000_000, 50_000_000]
for n in sizes:
    a_np = np.random.rand(n).astype(np.float32)
    b_np = np.random.rand(n).astype(np.float32)
    a_cp, b_cp = cp.asarray(a_np), cp.asarray(b_np)
    cpu_fn = lambda a=a_np, b=b_np: (a * 1.1 + b).sum()
    gpu_fn = lambda a=a_cp, b=b_cp: (a * 1.1 + b).sum()
    compare(f'N={n:,}', cpu_fn, gpu_fn, n_repeat=10, n_warmup=2)

# => 작은 N에서는 speedup<1(GPU가 느림), N이 커질수록 speedup이 커집니다.

<a id="3"></a>
## 3. 전송 포함 end-to-end 비용

GPU 연산 자체는 빨라도, 매번 결과를 `asnumpy`로 host에 가져오면 **전송이 병목**이 됩니다.
'연산만' vs '연산+전송'을 비교해 전송 비용을 체감합니다. (메모리·전송 최적화는 단원 3에서 심화)

In [ ]:
n = 10_000_000 # n을 변경해보세요.
a = cp.random.random(n, dtype=cp.float32)

r_compute = bench(lambda: (a * 2.0 + 1.0).sum(), n_repeat=20, name='compute_only')
r_e2e     = bench(lambda: cp.asnumpy((a * 2.0 + 1.0).sum()), n_repeat=20, name='compute+transfer')
print_bench(r_compute)
print_bench(r_e2e)
print('=> 결과 스칼라 하나를 가져오는 데도 전송 오버헤드가 붙습니다. 루프 안 반복 전송을 피하세요.')

<a id="4"></a>
## 4. 벡터화의 힘

GPU 가속의 **대전제**는 벡터화입니다. 파이썬 `for`로 원소를 하나씩 처리하면 NumPy의 최적화된 네이티브 연산을
활용하지 못하고, GPU에서는 특히 **수십~수백 배** 느려집니다. 항상 배열 연산으로 표현하세요.

<img src="images/figures/new_vectorize_vs_loop.png" width="600">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

<img src="images/figures/new_avoid_serial_loops.png" width="600">

<sub>그림: 본 코스 자체 제작 개념 도해</sub>

In [ ]:
import numba

# 같은 연산 A + A**2 : 파이썬 루프 vs 벡터화(CPU) vs 벡터화(GPU)
A = np.random.random((1024, 1024)).astype(np.float32) # n을 변경해보세요

def loop(A):
    out = np.empty_like(A)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            out[i, j] = A[i, j] + A[i, j] ** 2
    return out

@numba.njit(fastmath=True) # JIT 컴파일러에게 이 루프를 C수준으로 최적화하라고 명령!
def loop_fast(A):
    out = np.empty_like(A)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            out[i, j] = A[i, j] + A[i, j] ** 2
    return out

_ = loop_fast(A) # warm-up


t0 = time.perf_counter(); loop(A); t1 = time.perf_counter()
print(f'CPU 파이썬 루프 : {(t1 - t0) * 1e3:10.1f} ms')

t0 = time.perf_counter(); loop_fast(A); t1 = time.perf_counter()
print(f'NUMBA JIT 루프 : {(t1 - t0) * 1e3:10.1f} ms')

r_cpu = bench(lambda: A + A ** 2, n_repeat=10, name='cpu_vec')
print(f'CPU 벡터화      : {cpu_ms(r_cpu):10.3f} ms')

A_cp = cp.asarray(A)
r_gpu = bench(lambda: A_cp + A_cp ** 2, n_repeat=20, name='gpu_vec')
print(f'GPU 벡터화      : {cpu_ms(r_gpu):10.3f} ms')

<a id="5"></a>
## 5. 연습문제 — 손익분기 크기 찾기

`cp.sort`(GPU)와 `np.sort`(CPU)의 실행시간이 **역전되는 배열 크기**를 찾으세요.
- 여러 크기에 대해 `compare`로 CPU/GPU wall-clock을 측정합니다.
- speedup이 1을 넘어서기 시작하는 N을 보고합니다. (여유가 되면 matplotlib으로 그래프)

In [ ]:
sizes = [1_000, 10_000, 100_000, 1_000_000, 10_000_000, 50_000_000]

def find_breakeven(sizes):
    # TODO: 각 크기에서 np.sort vs cp.sort를 compare로 측정하고,
    #       speedup(>1)이 처음 나타나는 N을 반환하세요.
    # breakeven = None
    # ...
    # print('손익분기 크기(대략):', f'{breakeven:,}' if breakeven else '관측 범위 내 없음')
    # return breakeven
    raise NotImplementedError

# find_breakeven(sizes)

<details>
<summary>💡 해답 보기</summary>

```python
def find_breakeven(sizes):
    breakeven = None
    for n in sizes:
        y_np = np.random.rand(n).astype(np.float32)
        y_cp = cp.asarray(y_np)
        c, g, sp = compare(f'N={n:,}',
                           lambda a=y_np: np.sort(a),
                           lambda a=y_cp: cp.sort(a),
                           n_repeat=5, n_warmup=1)
        if breakeven is None and sp >= 1.0:
            breakeven = n
    print('손익분기 크기(대략):', f'{breakeven:,}' if breakeven else '관측 범위 내 없음')
    return breakeven

find_breakeven(sizes)
```

포인트: 작은 N은 런치·전송 오버헤드로 GPU가 느리고, N이 커지면 GPU가 역전합니다.
손익분기 크기는 **연산 종류·GPU 모델**에 따라 달라집니다.
</details>

## 🧪 추가 연습 & 실험

**연습 A — dtype 처리량**: 같은 연산 `(a*1.1+2).sum()` 을 `float32` vs `float64` GPU 배열로 측정해 시간비를 구하세요.
소비자용 GPU에서 float64는 보통 느립니다.

In [ ]:
def dtype_ratio(n=20_000_000):
    # TODO: a32(float32), a64(float64)에 같은 연산을 bench 하고 (f64시간/f32시간) 반환
    raise NotImplementedError

# print('f64/f32 시간비:', dtype_ratio())

<details><summary>💡 해답 보기</summary>

```python
def dtype_ratio(n=20_000_000):
    a32 = cp.random.random(n, dtype=cp.float32)
    a64 = cp.random.random(n, dtype=cp.float64)
    r32 = bench(lambda: (a32*1.1+2).sum(), n_repeat=20, name='f32')
    r64 = bench(lambda: (a64*1.1+2).sum(), n_repeat=20, name='f64')
    print_bench(r32); print_bench(r64)
    return gpu_ms(r64) / gpu_ms(r32)
print('f64/f32 시간비:', round(dtype_ratio(), 2))
```
</details>

**실험 B — matmul 손익분기**: 정사각 행렬 곱 `A@A` 에서 GPU가 CPU를 이기기 시작하는 N을 찾아보세요.
(예측 먼저: 원소별 연산보다 손익분기 N이 작을까 클까?)

In [ ]:
for N in [64, 128, 256, 512, 1024, 2048]:
    A_np = np.random.random((N, N)).astype(np.float32); A_cp = cp.asarray(A_np)
    compare(f'matmul N={N}', lambda A=A_np: A@A, lambda A=A_cp: A@A, n_repeat=5, n_warmup=2)

<a id="6"></a>
## 6. 체크포인트

- [ ] `bench`로 워밍업·동기화·반복평균이 적용된 시간을 측정할 수 있다
- [ ] 작은 배열에서 GPU가 더 느릴 수 있는 이유(런치 오버헤드·전송)를 설명할 수 있다
- [ ] '연산만' vs '연산+전송'의 차이를 수치로 확인했다
- [ ] 파이썬 루프 대신 벡터화를 써야 하는 이유를 안다
- [ ] 연습: `cp.sort`/`np.sort`의 손익분기 크기를 관측했다

다음: **`02_ndarray_core`** — ndarray의 구조·뷰/복사·브로드캐스팅과 NumPy→CuPy 포팅을 깊게 다룹니다.